In [2]:
from langgraph.graph import StateGraph
from typing import TypedDict

# Proper state schema
class State(TypedDict):
    number: int
    result: str

# nodes
def check_even(state):
    return state

def even_node(state):
    return {**state, "result": f"{state['number']} is even"}


def odd_node(state):
    return {**state, "result": f"{state['number']} is odd"}

# Graph
builder = StateGraph(State)

builder.add_node("check", check_even)
builder.add_node("even", even_node)
builder.add_node('odd', odd_node)

# Routing
def route(state):
    return "even" if state["number"]%2 == 0 else "odd"

builder.add_conditional_edges(
    "check",
    route,
    {
        "even": "even",
        "odd": "odd" 
    }
)

builder.set_entry_point("check")

graph = builder.compile()

# Run
print(graph.invoke({"number": 8}))


{'number': 8, 'result': '8 is even'}


# Multi Conditional Routing

In [12]:
from langgraph.graph import StateGraph
from typing import TypedDict

# Define Proper state schema
class State(TypedDict):
    score: int
    result: str

# nodes
def check_score(state):
    return state

def excellent_node(state):
    return {**state, "result": f"score {state['score']} -> EXCELLENT"}

def average_node(state):
    return {**state, "result": f"score {state['score']} -> AVERAGE"}

def fail_node(state):
    return {**state, "result": f"score {state['score']} -> FAIL"}

# Routing Logic

def route_score(state):
    score = state["score"]
    if score >= 90:
        return "excellent"
    elif score >= 50:
        return "average"
    else:
        return "fail"

# Build Graph

builder = StateGraph(State)

builder.add_node("check_score", check_score)
builder.add_node("excellent_node", excellent_node)
builder.add_node('average_node', average_node)
builder.add_node('fail_node', fail_node)

# Conditional Edges

builder.add_conditional_edges(
    "check_score",
    route_score,
    {
        "excellent": "excellent_node",
        "average": "average_node",
        "fail": "fail_node"
    }
)

#Entry points
builder.set_entry_point("check_score")

# (Optional but recommended) finish nodes
builder.set_finish_point("excellent_node")
builder.set_finish_point("average_node")
builder.set_finish_point("fail_node")

# compile
graph = builder.compile()

# Run Examples

print(graph.invoke({"score": 95}))
print(graph.invoke({"score": 70}))
print(graph.invoke({"score": 34}))


{'score': 95, 'result': 'score 95 -> EXCELLENT'}
{'score': 70, 'result': 'score 70 -> AVERAGE'}
{'score': 34, 'result': 'score 34 -> FAIL'}


In [1]:
from langgraph.graph import StateGraph
from typing import TypedDict

# State Schema
class State(TypedDict):
    attempt: int
    success: bool 
    result: str

# Node: Try Something
def try_node(state):
    attempt = state.get("attempt",0) +1

    #simulate success after 3 attempts
    success = attempt >= 3

    print(f"Attempt {attempt} -> {'SUCCESS' if success else 'FAIL'}")

    return {
        **state,
        "attempt":attempt,
        "success":success
    }

# Routing Logic
def check_success(state):
    return "end" if state["success"] else "retry"

# Build Graph
builder = StateGraph(State)

builder.add_node("try_node", try_node)

# conditional loop
builder.add_conditional_edges(
    "try_node",
    check_success,
    {
        "end":"__end__", # Stop execution
        "retry":"try_node" # loop again
    }
)

# Entry Point
builder.set_entry_point("try_node")

# compile
graph = builder.compile()


# Run
result = graph.invoke({
    "attempt":0,
    "success":False,
    "result": ""
})

print("\n Final Output: ")
print(result)




    

Attempt 1 -> FAIL
Attempt 2 -> FAIL
Attempt 3 -> SUCCESS

 Final Output: 
{'attempt': 3, 'success': True, 'result': ''}
